In [ ]:
# PyTorch 整体流程:
# 1. 数据加载
# 2. 模型搭建
# 3. 设计损失函数&优化器
# 4. 开始正式训练

In [ ]:
# 0.1 准备必要的python库 (包括PyTorch在内)
import torch
from torch import nn
from torchvision import datasets  # Datasets类
from torchvision.transforms import ToTensor, Normalize, Compose # 归一化，标准化，数据增强(组合)
from torch.utils.data import DataLoader # Dataloader数据加载器
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 0.2 确保有显存，支持GPU计算
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"当前使用的计算设备: {device}")

In [ ]:
# 1. 数据加载
# 为了让PyTorch能正常工作，需要"数据加载器"来将你的数据，转化为PyTorch框架能够处理的形式

In [ ]:
# 1.1 数据预处理：转为Tensor + 标准化
transform = Compose([
    ToTensor(),
    Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])
# 这里没有用crop, 因为原图尺寸和我们即将定义的网络输入尺寸，是一致的

# 1.2 下载数据集
print("正在准备 CIFAR-100 数据集...")
training_data = datasets.CIFAR100(
    root="data",
    train=True, # 训练集
    download=True,
    transform=transform
    # 神经网络计算只接受Tensor形式的输入, 需要把下载下来的PIL图像，转为Tensor
    # 思考：ToTensor之后的数据，是在cpu还是gpu上？
)

test_data = datasets.CIFAR100(
    root="data",
    train=False, # 验证集
    download=True,
    transform=transform
)

# 1.3 创建DataLoader
batch_size=64

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# 打印数据维度 (搭建模型时要用到)
for X, y in test_dataloader:
    print(f"输入 X 的维度 [B, C, H, W]: {X.shape}")
    print(f"标签 y 的维度: {y.shape}")
    print(f"标签 y 的值: {y}")
    break

In [ ]:
# 随机取一个batch的前4张图, 做可视化
def imshow(img):
    img = img / 2 + 0.5  # 逆标准化，用于显示
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

dataiter = iter(train_dataloader)
images, labels = next(dataiter)
plt.figure(figsize=(10, 3))
for i in range(4):
    plt.subplot(1, 4, i+1)
    imshow(images[i])
plt.show()

## Tensor 基础操作

In [ ]:
# --- 创建 Tensor ---
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
b = torch.zeros(2, 3)
c = torch.ones(2, 3)
d = torch.randn(2, 3)
e = torch.arange(0, 12).reshape(3, 4)

print("a =", a)
print("zeros =", b)
print("randn =", d)
print("arange+reshape =", e)

In [ ]:
# --- 属性 ---
print("shape:", a.shape)      # 各维度大小
print("dtype:", a.dtype)      # 数据类型
print("device:", a.device)    # 所在设备 (cpu/cuda)
print("ndim:", a.ndim)        # 维度数
print("numel:", a.numel())    # number of elements, 元素总数 = 2*3 = 6

In [ ]:
# --- 数学运算与归约 ---
x = torch.tensor([[1., 2.], [3., 4.]])

print("逐元素加:", x + 10)
print("逐元素乘:", x * x)
print("矩阵乘法:", x @ x)
print("转置:    ", x.T)

# 归约操作: sum, mean, max
print("\n全局求和:", x.sum())
print("按行求和:", x.sum(dim=1))         # dim=1 沿列方向压缩, 每行得一个值
print("按列最大:", x.max(dim=0))

# mean + 多维度归约
img = torch.randn(3, 4, 4)               # 模拟 [C, H, W]
print("\nimg shape:", img.shape)
print("全局 mean:", img.mean())
print("按通道 mean:", img.mean(dim=0).shape)         # [4, 4] — 沿 C 维压缩
print("空间 mean:", img.mean(dim=(1, 2)).shape)       # [3]   — 沿 H,W 两维压缩, 得每通道均值
print("空间 mean 值:", img.mean(dim=(1, 2)))

batch = torch.randn(8, 3, 4, 4)          # [B, C, H, W]
print("\nbatch 每图每通道均值:", batch.mean(dim=(2, 3)).shape)  # [8, 3]

In [ ]:
# --- 索引与切片 ---
t = torch.arange(24).reshape(2, 3, 4)
print("shape:", t.shape)
print("t[0]:", t[0].shape)
print("t[:, 1]:", t[:, 1])
print("t[..., -1]:", t[..., -1])

# 布尔索引
vals = torch.tensor([3, 1, 4, 1, 5, 9])
print("大于3:", vals[vals > 3])

In [ ]:
# --- 维度变换 ---
img = torch.randn(3, 32, 32)

print("原始:     ", img.shape)
print("unsqueeze:", img.unsqueeze(0).shape)      # 加 batch 维
print("squeeze:  ", img.unsqueeze(0).squeeze(0).shape)
print("permute:  ", img.permute(1, 2, 0).shape)  # CHW -> HWC
print("transpose:", img.transpose(0, 2).shape)    # 交换第 0 和第 2 维

# --- reshape vs view vs contiguous ---
# view: 要求内存连续, 不拷贝数据, 更快
# reshape: 自动处理, 非连续时会拷贝, 更安全
print("\nreshape:", img.reshape(3, -1).shape)
print("view:   ", img.view(3, -1).shape)

# permute/transpose 后内存不连续, view 会报错
t = img.permute(1, 2, 0)
print("\npermute 后 is_contiguous:", t.is_contiguous())
# t.view(-1)         # 会报错!
print("contiguous().view:", t.contiguous().view(-1).shape)  # 先变连续再 view
print("reshape 直接可用: ", t.reshape(-1).shape)             # reshape 自动处理

# flatten: reshape(-1) 的快捷方式
print("flatten:", img.flatten().shape)

In [ ]:
# --- 拼接: cat vs stack ---
a = torch.randn(2, 3)
b = torch.randn(2, 3)

# cat: 沿已有维度拼接
print("cat dim=0:", torch.cat([a, b], dim=0).shape)   # [4, 3]
print("cat dim=1:", torch.cat([a, b], dim=1).shape)   # [2, 6]

# stack: 新建维度再拼
print("stack dim=0:", torch.stack([a, b], dim=0).shape)  # [2, 2, 3]

# 实际用途: 多张图 stack 成 batch
imgs = [torch.randn(3, 32, 32) for _ in range(4)]
print("batch:", torch.stack(imgs).shape)  # [4, 3, 32, 32]

### einsum: 爱因斯坦求和约定

`torch.einsum(equation, *tensors)` 用下标字符串描述任意张量运算，规则：

1. 每个输入张量分配若干下标字母，如 `"ij"` 表示二维矩阵的行 `i` 和列 `j`
2. `->` 右侧指定输出保留哪些下标；**未出现在右侧的下标会被求和消掉**
3. 同一字母在不同输入中出现，表示沿该维度做乘法再求和（即缩并/contraction）

常见模式速查：

| 表达式 | 含义 | 等价操作 |
|---|---|---|
| `ij,jk->ik` | 矩阵乘法 | `A @ B` |
| `bij,bjk->bik` | batch 矩阵乘法 | `torch.bmm(A, B)` |
| `ij->ji` | 转置 | `A.T` |
| `ii->` | 矩阵的迹 (trace) | `torch.trace(A)` |
| `ij,ij->` | Frobenius 内积 | `(A * B).sum()` |
| `bs,bsd->bd` | 加权求和 | 注意力机制中常见 |

In [ ]:
# --- einsum ---
A = torch.randn(2, 3)
B = torch.randn(3, 4)

# 矩阵乘法
C = torch.einsum("ij,jk->ik", A, B)
print("matmul:", C.shape)

# batch 矩阵乘法
X = torch.randn(8, 3, 4)
Y = torch.randn(8, 4, 5)
Z = torch.einsum("bij,bjk->bik", X, Y)
print("batch matmul:", Z.shape)

# 转置
print("转置:", torch.einsum("ij->ji", A).shape)

# trace
S = torch.randn(3, 3)
print("trace:", torch.einsum("ii->", S).item())

# 注意力加权求和: [B, seq] x [B, seq, dim] -> [B, dim]
w = torch.randn(8, 10)
v = torch.randn(8, 10, 64)
print("加权求和:", torch.einsum("bs,bsd->bd", w, v).shape)

In [ ]:
# --- GPU 迁移 ---
t = torch.randn(3, 3)
print("初始 device:", t.device)

# NVIDIA GPU:
# if torch.cuda.is_available():
#     t_gpu = t.to("cuda")

# Apple Silicon (M1/M2/M3) 使用 MPS 后端:
if torch.backends.mps.is_available():
    t_gpu = t.to("mps")
    print("MPS device:", t_gpu.device)
    print("取回 CPU:", t_gpu.cpu().device)
else:
    print("无 GPU, 跳过")

### 练习 1: Tensor 操作综合 (5 min)

完成以下操作, 将结果赋值给指定变量名, 然后运行校验 cell:

In [ ]:
torch.manual_seed(42)
data = torch.randn(8, 3, 32, 32)  # 8 张模拟图像

# 1. 用切片取前 4 张和后 4 张
batch_a = ...  # shape: [4, 3, 32, 32]
batch_b = ...  # shape: [4, 3, 32, 32]

# 2. 将 batch_a 和 batch_b 沿宽度方向(dim=3)拼接
wide = ...     # shape: [4, 3, 32, 64]

# 3. 对 data 求每张图像各通道的空间平均值 (在 H, W 两个维度上 mean)
channel_means = ...  # shape: [8, 3]

# 4. 用 einsum 对 channel_means 做转置
means_T = ...  # shape: [3, 8]

# 5. 将 wide 展平, 保留 batch 维
flat = ...     # shape: [4, 6144]

In [ ]:
# ---- 练习 1 校验 ----
torch.manual_seed(42)
_data = torch.randn(8, 3, 32, 32)

assert batch_a.shape == torch.Size([4, 3, 32, 32]), f"batch_a shape 错误: {batch_a.shape}"
assert torch.equal(batch_a, _data[:4]), "batch_a 值不对, 检查切片方式"

assert wide.shape == torch.Size([4, 3, 32, 64]), f"wide shape 错误: {wide.shape}"
assert torch.equal(wide, torch.cat([_data[:4], _data[4:]], dim=3)), "wide 拼接不对"

assert channel_means.shape == torch.Size([8, 3]), f"channel_means shape 错误: {channel_means.shape}"
assert torch.allclose(channel_means, _data.mean(dim=(2, 3))), "channel_means 计算不对"

assert means_T.shape == torch.Size([3, 8]), f"means_T shape 错误: {means_T.shape}"
assert torch.allclose(means_T, channel_means.T), "means_T 转置不对"

assert flat.shape == torch.Size([4, 6144]), f"flat shape 错误: {flat.shape}"

print("ALL PASSED")
print(f"  channel_means[0]: {channel_means[0].tolist()}")
print(f"  flat checksum: {flat.sum().item():.2f}")

## 常见网络层详解

| 层 | 作用 | 关键参数 |
|---|---|---|
| `Conv2d` | 卷积提取空间特征 | in/out channels, kernel_size, padding, stride, dilation, groups |
| `BatchNorm2d` | 逐通道归一化 | num_features |
| `ReLU` / `GELU` | 非线性激活 | — |
| `MaxPool2d` / `AdaptiveAvgPool2d` | 下采样 | kernel_size / output_size |
| `Flatten` | 展平为一维 | — |
| `Linear` | 全连接 | in_features, out_features |
| `Dropout` | 随机置零防过拟合 | p |

In [ ]:
# --- Conv2d 基础: kernel_size, padding ---
x = torch.randn(1, 3, 32, 32)
print("输入:", x.shape)

# k=3, p=1: 尺寸不变. 公式: H_out = (H - k + 2p) / s + 1
print("k=3,p=1:", nn.Conv2d(3, 64, 3, padding=1)(x).shape)   # [1,64,32,32]
print("k=5,p=2:", nn.Conv2d(3, 64, 5, padding=2)(x).shape)   # [1,64,32,32]
print("k=3,p=0:", nn.Conv2d(3, 64, 3, padding=0)(x).shape)   # [1,64,30,30]

In [ ]:
# --- Conv2d 进阶: stride, dilation ---
x = torch.randn(1, 3, 32, 32)

# stride=2: 步长为2, 输出尺寸减半
conv_s2 = nn.Conv2d(3, 64, 3, padding=1, stride=2)
print("stride=2:", conv_s2(x).shape)   # [1,64,16,16]

# dilation=2: 空洞卷积, 感受野从3x3扩到5x5, 参数量不变
conv_d2 = nn.Conv2d(3, 64, 3, padding=2, dilation=2)
print("dilation=2:", conv_d2(x).shape)  # [1,64,32,32]

print(f"两者参数量相同: {sum(p.numel() for p in conv_s2.parameters())} == {sum(p.numel() for p in conv_d2.parameters())}")

In [ ]:
# --- 动手试试 ---
x = torch.randn(1, 3, 32, 32)

# 挑战: 用一个 Conv2d 把 [1,3,32,32] 变成 [1,32,8,8]
# 提示: H_out = (H_in - k + 2p) / s + 1, 需要 s > 1
# your_conv = nn.Conv2d(3, 32, kernel_size=?, padding=?, stride=?)
# print("输出:", your_conv(x).shape)

# 试试: dilation=3 时, 等效感受野多大? 需要多少 padding 才能保持尺寸?
# hint: 等效 k = k + (k-1)*(d-1)

In [ ]:
# --- Conv2d groups: 分组卷积 ---
x = torch.randn(1, 64, 16, 16)

# groups=1 普通卷积
conv_normal = nn.Conv2d(64, 64, 3, padding=1, groups=1)
# groups=64 深度卷积 (depthwise): 每个通道独立卷积
conv_dw = nn.Conv2d(64, 64, 3, padding=1, groups=64)
# pointwise 1x1 卷积: 跨通道混合
conv_pw = nn.Conv2d(64, 128, 1)

print(f"普通 Conv 参数量:       {sum(p.numel() for p in conv_normal.parameters())}")
print(f"Depthwise 参数量:       {sum(p.numel() for p in conv_dw.parameters())}")
print(f"Depthwise+Pointwise:    {sum(p.numel() for p in conv_dw.parameters()) + sum(p.numel() for p in conv_pw.parameters())}")
print(f"等效普通 Conv(64->128): {sum(p.numel() for p in nn.Conv2d(64, 128, 3, padding=1).parameters())}")
print()
out = conv_pw(conv_dw(x))
print("Depthwise separable 输出:", out.shape)

In [ ]:
# --- BatchNorm2d ---
x = torch.randn(4, 64, 16, 16)
bn = nn.BatchNorm2d(64)

bn.train()
y = bn(x)
print("训练模式 mean~0:", y.mean(dim=(0,2,3))[:4].tolist())

bn.eval()
_ = bn(x)
print("running_mean[:4]:", bn.running_mean[:4].tolist())

In [ ]:
# --- 激活函数对比 + 可视化 ---
x_pts = torch.linspace(-3, 3, 7)
print("采样点:", x_pts.tolist())
print("ReLU:     ", F.relu(x_pts).tolist())
print("LeakyReLU:", F.leaky_relu(x_pts, 0.1).tolist())
print("GELU:     ", [round(v, 2) for v in F.gelu(x_pts).tolist()])
print("Sigmoid:  ", [round(v, 2) for v in torch.sigmoid(x_pts).tolist()])

# 绘制平滑曲线 + 标记采样点
x_smooth = torch.linspace(-4, 4, 200)
funcs = {
    "ReLU": F.relu,
    "LeakyReLU(0.1)": lambda t: F.leaky_relu(t, 0.1),
    "GELU": F.gelu,
    "Sigmoid": torch.sigmoid,
}

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, (name, fn) in zip(axes, funcs.items()):
    y_smooth = fn(x_smooth)
    y_pts = fn(x_pts)
    ax.plot(x_smooth.numpy(), y_smooth.numpy(), linewidth=2)
    ax.scatter(x_pts.numpy(), y_pts.numpy(), color="red", s=50, zorder=5)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name, fontsize=12)
    ax.set_xlim(-4, 4)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 池化层 ---
x = torch.randn(1, 64, 16, 16)

print("MaxPool2d(2):        ", nn.MaxPool2d(2)(x).shape)           # [1,64,8,8]
print("AvgPool2d(2):        ", nn.AvgPool2d(2)(x).shape)           # [1,64,8,8]
print("AdaptiveAvgPool2d(1):", nn.AdaptiveAvgPool2d(1)(x).shape)   # [1,64,1,1] 全局平均池化

In [ ]:
# --- 动手试试 ---
x = torch.randn(1, 64, 32, 32)

# 1. 用 MaxPool2d 一步把 32x32 降到 8x8 (提示: kernel_size=?)
# y1 = nn.MaxPool2d(?)(x)
# print("MaxPool ->", y1.shape)

# 2. 用 AdaptiveAvgPool2d 直接输出 7x7 (ResNet 风格)
# y2 = nn.AdaptiveAvgPool2d(?)(x)
# print("AdaptiveAvg ->", y2.shape)

# 3. 如果输入是 [1, 64, 15, 15] 这种奇数尺寸, MaxPool2d(2) 会怎样?
odd = torch.randn(1, 64, 15, 15)
# print("奇数尺寸 MaxPool(2):", nn.MaxPool2d(2)(odd).shape)

In [ ]:
# --- Flatten + Linear + Dropout ---
x = torch.randn(1, 256, 4, 4)

x = nn.Flatten()(x)
print("Flatten:        ", x.shape)         # [1, 4096]

x = nn.Linear(4096, 512)(x)
x = nn.ReLU()(x)
print("Linear+ReLU:    ", x.shape)         # [1, 512]

# Dropout 演示: 用全 1 向量避免 ReLU 零值干扰
demo = torch.ones(1, 512)
drop = nn.Dropout(0.5)

drop.train()
out_train = drop(demo)
print(f"训练时置零率:    {(out_train == 0).float().mean():.0%}")  # ~50%

drop.eval()
out_eval = drop(demo)
print(f"推理时置零率:    {(out_eval == 0).float().mean():.0%}")   # 0%
# 训练时非零值会乘以 1/(1-p)=2 做补偿, 保持期望不变
print(f"训练时非零值:    {out_train[out_train > 0][0].item():.1f} (原始 1.0 * 2)")

x = nn.Linear(512, 100)(x)
print("输出层:          ", x.shape)         # [1, 100]

### nn.Sequential

将上面的层按顺序打包，省去逐层调用。下一节的 `SimpleCNN` 正是这样组织的。

```python
self.features = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    ...
)
```

In [ ]:
# 2.1 模型定义

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential( # backbone
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 尺寸变为 16x16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 尺寸变为 8x8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 尺寸变为 4x4
        )
        self.classifier = nn.Sequential( # head / decoder
            nn.Flatten(), # (h,w,c) → (h*w*c, 1) 多维 展平到 一维
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 100) # CIFAR-100的100个类别
        )

    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits

In [ ]:
# 2.2 模型实例化
model = SimpleCNN().to(device)
print(model)

## 损失函数、优化器与反向传播

模型定义好了，接下来要回答三个问题：
1. **怎么衡量预测有多差？** — 损失函数
2. **参数该往哪个方向调？** — 反向传播 (autograd)
3. **怎么调整参数？** — 优化器

### 三种常见损失函数

| 损失函数 | 适用场景 | 输入 | 公式 |
|---|---|---|---|
| `MSELoss` | 回归 | 预测值 vs 真实值 | $L = \frac{1}{N}\sum_{i}(y_i - \hat{y}_i)^2$ |
| `CrossEntropyLoss` | 多分类 | logits `[B, C]` + 类别索引 `[B]` | $L = -\frac{1}{N}\sum_{i}\log\frac{e^{z_{i,y_i}}}{\sum_j e^{z_{i,j}}}$ |
| `BCEWithLogitsLoss` | 二分类 / 多标签 | logits `[B]` + 标签 `{0,1}` | $L = -\frac{1}{N}\sum_{i}[y_i\log\sigma(z_i) + (1-y_i)\log(1-\sigma(z_i))]$ |

其中 $\sigma(z) = \frac{1}{1+e^{-z}}$ 是 sigmoid 函数。

In [ ]:
# --- MSELoss: 均方误差 ---
# 回归任务: 预测一个连续数值
pred = torch.tensor([2.5, 0.0, 2.1])
target = torch.tensor([3.0, -0.5, 2.0])

# PyTorch 实现
mse_loss = nn.MSELoss()
loss_pt = mse_loss(pred, target)
print("PyTorch MSELoss:", loss_pt.item())

# 手动实现: mean((pred - target)^2)
loss_manual = ((pred - target) ** 2).mean()
print("手动实现:       ", loss_manual.item())
print("一致:", torch.allclose(loss_pt, loss_manual))

In [ ]:
# --- CrossEntropyLoss: 多分类 ---
# 输入: logits (未经 softmax), 标签为类别索引 (不是 one-hot)
logits = torch.tensor([[2.0, 0.5, 0.1],    # 样本 0
                        [0.1, 0.3, 3.0]])    # 样本 1
labels = torch.tensor([0, 2])                # 真实类别

ce_loss = nn.CrossEntropyLoss()
loss_pt = ce_loss(logits, labels)
print("PyTorch CrossEntropyLoss:", loss_pt.item())

# 手动实现: softmax -> log -> 取对应类别 -> 取负平均
probs = torch.softmax(logits, dim=1)
print("softmax 概率:", probs)
log_probs = torch.log(probs)
loss_manual = -(log_probs[0, labels[0]] + log_probs[1, labels[1]]) / 2
print("手动实现:                ", loss_manual.item())
print("一致:", torch.allclose(loss_pt, loss_manual))

### 练习: 手动实现 BCEWithLogitsLoss

`BCEWithLogitsLoss` 在内部先对 logits 做 sigmoid，再计算二元交叉熵：

$$L = -\frac{1}{N}\sum_{i}\left[y_i \cdot \log(\sigma(z_i)) + (1 - y_i) \cdot \log(1 - \sigma(z_i))\right]$$

请用 `torch.sigmoid` 和 `torch.log` 手动实现，使结果与 `nn.BCEWithLogitsLoss()` 一致。

**提示**: 注意数值稳定性，`log(0)` 会产生 `-inf`。可以用 `torch.clamp(x, min=1e-7)` 避免。

In [ ]:
# 给定数据
torch.manual_seed(0)
logits = torch.tensor([2.0, -1.0, 0.5, -0.3, 3.0])
labels = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])

# PyTorch 标准实现
bce_loss = nn.BCEWithLogitsLoss()
loss_pt = bce_loss(logits, labels)
print("PyTorch BCEWithLogitsLoss:", loss_pt.item())

# TODO: 手动实现
# 1. 对 logits 做 sigmoid
# 2. 计算二元交叉熵
# 3. 取平均

# bce_manual_loss = ...
# print("手动实现:", bce_manual_loss.item())

In [ ]:
# ---- BCE 练习校验 ----
ref_loss = nn.BCEWithLogitsLoss()(
    torch.tensor([2.0, -1.0, 0.5, -0.3, 3.0]),
    torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])
)
assert torch.allclose(bce_manual_loss, ref_loss, atol=1e-5),     f"结果不一致: 你的 {bce_manual_loss.item():.6f}, 期望 {ref_loss.item():.6f}"
print("PASSED")
print(f"  你的结果: {bce_manual_loss.item():.6f}")
print(f"  期望结果: {ref_loss.item():.6f}")

In [ ]:
# 3. 损失函数 & 优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# 4.1 训练函数
loss_history = []

def train(dataloader, model, loss_fn, optimizer, loss_history):
  size = len(dataloader.dataset)
  model.train()
  
  for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)

      pred = model(X) # 前向传播
      loss = loss_fn(pred, y) # 计算损失

      optimizer.zero_grad() # 梯度清零
      loss.backward() # 反向传播
      optimizer.step() # 更新参数

      if batch % 100 == 0:
          loss, current = loss.item(), (batch + 1) * len(X)
          loss_history.append(loss) # 记录loss
          print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

### 测试评估中的关键操作

训练函数中出现了 `model.train()`，测试函数中则用 `model.eval()` + `torch.no_grad()`。
此外，计算准确率用到了 `argmax` 和布尔比较链，下面逐一拆解。

In [ ]:
# --- model.eval() vs model.train() ---
# train(): Dropout 随机丢弃, BN 用当前 batch 统计量
# eval():  Dropout 关闭, BN 用 running_mean/running_var
# 测试时必须切换, 否则结果不稳定

# --- torch.no_grad() ---
# 测试时不需要计算梯度, 用 no_grad() 可以:
# 1. 节省显存 (不存储中间激活值)
# 2. 加速计算
a = torch.tensor([2.0], requires_grad=True)
b = a * 3
print("有梯度:", b.requires_grad)       # True

with torch.no_grad():
    c = a * 3
    print("no_grad:", c.requires_grad)   # False

# --- argmax: 取最大值的索引 ---
logits = torch.tensor([[0.1, 0.3, 2.5, 0.8],   # 第 0 个样本, 最大在 index=2
                        [1.2, 0.1, 0.3, 0.05]])  # 第 1 个样本, 最大在 index=0

print("\nlogits shape:", logits.shape)            # [2, 4]
print("argmax(dim=1):", logits.argmax(dim=1))     # tensor([2, 0]) — 每行取最大位置

# --- 准确率计算链拆解 ---
labels = torch.tensor([2, 1])                     # 真实标签

preds = logits.argmax(dim=1)                      # tensor([2, 0])
correct_mask = (preds == labels)                   # tensor([True, False])
correct_float = correct_mask.type(torch.float)     # tensor([1., 0.])
num_correct = correct_float.sum().item()           # 1.0 (标量)

print("预测:", preds.tolist())
print("标签:", labels.tolist())
print("逐个对比:", correct_mask.tolist())
print("正确数:", num_correct)
print("准确率:", f"{num_correct / len(labels):.0%}")

In [ ]:
# 4.2 测试函数
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0, 0
  with torch.no_grad():
      for X, y in dataloader:
          X, y = X.to(device), y.to(device)
          pred = model(X)
          test_loss += loss_fn(pred, y).item()
          correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
# 4.3 核心训练过程

# (这里我们每训1个epoch，就测试1次accuracy)

epochs = 15
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, loss_history)
    test(test_dataloader, model, loss_fn) # loss_fn在此是为了打印出loss, 实际不需要
print("Done!")

# 打印Loss曲线
plt.plot(loss_history)
plt.xlabel('Batch (x100)')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# 保存model
torch.save(model.state_dict(), "SimpleCNN.pth")

In [ ]:
# 5. 取样本，作测试
# classes 顺序必须与 CIFAR-100 的 fine label 索引一致 (字母序)
classes = [
    '苹果', '观赏鱼', '婴儿', '熊', '海狸',
    '床', '蜜蜂', '甲虫', '自行车', '瓶子',
    '碗', '男孩', '桥', '公共汽车', '蝴蝶',
    '骆驼', '罐头', '城堡', '毛毛虫', '牛',
    '椅子', '黑猩猩', '时钟', '云', '蟑螂',
    '沙发', '螃蟹', '鳄鱼', '杯子', '恐龙',
    '海豚', '大象', '比目鱼', '森林', '狐狸',
    '女孩', '仓鼠', '房屋', '袋鼠', '电脑键盘',
    '灯', '割草机', '豹', '狮子', '蜥蜴',
    '龙虾', '男人', '枫树', '摩托车', '山',
    '老鼠', '蘑菇', '橡树', '橙子', '兰花',
    '水獭', '棕榈树', '梨', '皮卡车', '松树',
    '平原', '盘子', '罂粟', '豪猪', '负鼠',
    '兔子', '浣熊', '鳐鱼', '道路', '火箭',
    '玫瑰', '海洋', '海豹', '鲨鱼', '鼩鼱',
    '臭鼬', '摩天大楼', '蜗牛', '蛇', '蜘蛛',
    '松鼠', '有轨电车', '向日葵', '甜椒', '桌子',
    '坦克', '电话', '电视', '老虎', '拖拉机',
    '火车', '鳟鱼', '郁金香', '乌龟', '衣柜',
    '鲸鱼', '柳树', '狼', '女人', '蠕虫',
]

# 5.1 随机挑选一个test sample，可视化
x, y = test_data[3][0], test_data[3][1]
x_display = x.permute(1, 2, 0)

mean = torch.tensor([0.5071, 0.4867, 0.4408])
std = torch.tensor([0.2675, 0.2565, 0.2761])
x_display = x_display * std + mean
x_display = torch.clamp(x_display, 0, 1)

plt.imshow(x_display)
plt.title(f"Label: {classes[y]} (index {y})")
plt.axis("off")
plt.show()

print("真实标签:", classes[y], f"(index {y})")

In [ ]:
# 5.2 核心推理
model.eval()
with torch.no_grad():
    input_tensor = x.unsqueeze(0).to(device)
    pred = model(input_tensor)
    idx = pred[0].argmax(0).item()
    predicted = classes[idx]
    actual = classes[y]

print(f"预测结果: {predicted}")
print(f"真实标签: {actual}")